# Train Visualization - Rejseplanen NeTEx

Visualize train departures and routes for a selected day using Rejseplanen NeTEx data and AnyMap-TS mapping.

## Overview
- **Data Source:** Rejseplanen+NeTEx.zip (Combined public transport schedule data for Denmark)
- **Filter:** Trains only (DSB lines, excludes metro and S-trains)
- **Interactive Map:** Visualize train movements in real-time on an interactive MapLibre map
- **Date Selection:** Choose any day within the data's valid interval
- **Controls:** Start, Pause, Resume, and Stop buttons allow you to control playback without rerunning cells

## How It Works
1. This notebook loads NeTEx XML files from a ZIP archive
2. Parses line metadata, stops, and service journeys from XML
3. Builds interpolatable movement segments between consecutive stops
4. Runs a discrete-time simulation loop that updates train positions every second
5. Displays trains on the map using `LiveMapLibreMap.move_marker()` for efficient updates

## Documentation
- For detailed architecture and usage, see [docs/train_simulation.md](../docs/train_simulation.md)
- For NeTEx XML format details, see [docs/netex_format.md](../docs/netex_format.md)

## Quick Start
1. Run all cells (they will execute in order)
2. Select a date from the date picker widget
3. Click **Start** to begin the simulation
4. Use **Pause**, **Resume**, and **Stop** to control playback
5. Observe trains moving on the map as they travel between stops

## Section 1: Import Required Libraries

In [1]:
import pandas as pd
import json
import zipfile
from pathlib import Path
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional
import xml.etree.ElementTree as ET
from collections import defaultdict

# Import AnyMap-TS live map helper for in-place marker movement
from simulated_city.maplibre_live import LiveMapLibreMap

print("✓ Dependencies imported successfully")

✓ Dependencies imported successfully


## Section 2: Extract and Configure NeTEx Data Path

In [2]:
# Find and configure the NeTEx ZIP file
zip_path = Path("../Rejseplanen+NeTEx.zip")

if not zip_path.exists():
    # Try from workspace root
    zip_path = Path(".").resolve().parent / "Rejseplanen+NeTEx.zip"

if not zip_path.exists():
    raise FileNotFoundError(f"Could not find Rejseplanen+NeTEx.zip at {zip_path}")

print(f"✓ Found NeTEx data: {zip_path}")

# List available operators in the ZIP
with zipfile.ZipFile(zip_path) as z:
    files = z.namelist()
    # Extract unique operators (e.g., DSB_000002, MET_Metro)
    operators = set()
    for f in files:
        parts = f.split('/')
        if len(parts) > 0:
            operators.add(parts[0])
    
print(f"✓ Available operators: {sorted(operators)}")
print(f"  Train operators (DSB): Lines like 001, 002, 022, etc.")
print(f"  Metro operator (MET): Copenhagen Metro M1, M2, M3, M4")
print(f"\nFiltering for trains only (DSB lines, no metro)")

# We will filter only DSB (trains), not MET (metro)
TRAIN_OPERATORS = [op for op in operators if op.startswith("DSB_")]
print(f"✓ Selected {len(TRAIN_OPERATORS)} train operator(s) for visualization")

✓ Found NeTEx data: ../Rejseplanen+NeTEx.zip
✓ Available operators: ['AEROE_AEROE', 'AErF_616', 'AErF_617', 'AErF_620', 'AErX_619', 'BAT_BAT', 'DSB_000002', 'DSB_000022', 'DSB_001002', 'DSB_001022', 'DSB_112___', 'FYNBUS_FYNBUS', 'FYNBUS_Letbane', 'GoC_000087', 'GoC_000088', 'HL_HL', 'MET_000083', 'MET_Metro', 'MIDT_Aar', 'MIDT_Ebe', 'MIDT_Fav', 'MIDT_Gre', 'MIDT_Her', 'MIDT_Hor', 'MIDT_Ika', 'MIDT_LET', 'MIDT_MT', 'MIDT_Odd', 'MIDT_Ran', 'MIDT_Sil', 'MIDT_Ska', 'MIDT_Str', 'MIDT_Vib', 'MOVIA_LOKT', 'MOVIA_MOVIA', 'NT_AAL', 'NT_FRH', 'NT_HO', 'NT_Hol', 'NT_NT', 'NT_NT-LAE', 'NT_NT-TID', 'NT_NYK', 'NT_THI', 'Nuup_Nuup', 'SAM_651', 'SAM_661', 'SYD_Aab', 'SYD_Esb', 'SYD_Frd', 'SYD_Had', 'SYD_Kol', 'SYD_ST', 'SYD_Sdb', 'SYD_SydVMS', 'SYD_Vjl', 'Skaane_001202']
  Train operators (DSB): Lines like 001, 002, 022, etc.
  Metro operator (MET): Copenhagen Metro M1, M2, M3, M4

Filtering for trains only (DSB lines, no metro)
✓ Selected 5 train operator(s) for visualization


## Section 3: Parse NeTEx XML for Routes and Stops

This cell contains the core **NeTEx parser**. It reads a single XML file and extracts:

- **Line metadata**: Name, public code (e.g., "IC 2"), operator, transport mode
- **Stops**: All train station locations with latitude/longitude coordinates
- **Service journeys**: The timetables for scheduled train services on a given day

The parser supports **two timing models**:
- **Model A** (`calls/Call`): Legacy format with inline arrival/departure times
- **Model B** (`passingTimes/TimetabledPassingTime`): Modern DSB format using references

For detailed information on NeTEx structure and parser logic, see [docs/netex_format.md](../docs/netex_format.md).

In [3]:
def parse_netex_xml(xml_content: str, operator: str) -> Dict:
    """Parse NeTEx XML and extract line, stops, and schedule information.

    Supports both:
    - ServiceJourney/calls/Call
    - ServiceJourney/passingTimes/TimetabledPassingTime
    """
    try:
        root = ET.fromstring(xml_content)
    except ET.ParseError:
        return None

    ns = {'nx': 'http://www.netex.org.uk/netex'}

    result = {
        'operator': operator,
        'operator_name': None,
        'line_number': None,
        'line_public_code': None,
        'line_name': None,
        'line_transport_mode': None,
        'stops': {},
        'journeys': [],
        'valid_from': None,
        'valid_to': None,
    }

    # Operator name (useful for filtering S-train feeds)
    op_name_elem = root.find('.//nx:Operator/nx:Name', ns)
    if op_name_elem is not None and op_name_elem.text:
        result['operator_name'] = op_name_elem.text.strip()

    # Line metadata
    line_elem = root.find('.//nx:Line', ns)
    if line_elem is not None:
        line_name_elem = line_elem.find('nx:Name', ns)
        public_code_elem = line_elem.find('nx:PublicCode', ns)
        mode_elem = line_elem.find('nx:TransportMode', ns)

        if line_name_elem is not None and line_name_elem.text:
            result['line_name'] = line_name_elem.text.strip()
        if public_code_elem is not None and public_code_elem.text:
            result['line_public_code'] = public_code_elem.text.strip()
        if mode_elem is not None and mode_elem.text:
            result['line_transport_mode'] = mode_elem.text.strip().lower()

    line_ref = root.find('.//nx:LineRef', ns)
    if line_ref is not None:
        line_id = line_ref.get('ref', '')
        if '::Line:' in line_id:
            result['line_number'] = line_id.split('::Line:')[1].split('::')[0]

    valid_between = root.find('.//nx:ValidBetween', ns)
    if valid_between is not None:
        from_date = valid_between.find('nx:FromDate', ns)
        to_date = valid_between.find('nx:ToDate', ns)
        if from_date is not None and from_date.text:
            result['valid_from'] = from_date.text[:10]
        if to_date is not None and to_date.text:
            result['valid_to'] = to_date.text[:10]

    # Scheduled stop points with coordinates
    for stop_elem in root.findall('.//nx:ScheduledStopPoint', ns):
        stop_id = stop_elem.get('id', '')
        name_elem = stop_elem.find('nx:Name', ns)
        name = name_elem.text if name_elem is not None else 'Unknown'

        loc_elem = stop_elem.find('nx:Location', ns)
        if loc_elem is not None:
            lat_elem = loc_elem.find('nx:Latitude', ns)
            lng_elem = loc_elem.find('nx:Longitude', ns)
            if lat_elem is not None and lng_elem is not None:
                try:
                    result['stops'][stop_id] = {
                        'name': name,
                        'lat': float(lat_elem.text),
                        'lng': float(lng_elem.text),
                    }
                except (ValueError, TypeError):
                    pass

    # Map StopPointInJourneyPattern -> ScheduledStopPoint
    spjp_to_stop_id = {}
    for spjp in root.findall('.//nx:StopPointInJourneyPattern', ns):
        spjp_id = spjp.get('id', '')
        stop_ref = spjp.find('nx:ScheduledStopPointRef', ns)
        if stop_ref is not None:
            stop_id = stop_ref.get('ref', '')
            if spjp_id and stop_id:
                spjp_to_stop_id[spjp_id] = stop_id

    # Service journeys
    for journey in root.findall('.//nx:ServiceJourney', ns):
        journey_id = journey.get('id', '')
        journey_stops = []

        # Variant A: calls/Call
        call_nodes = journey.findall('nx:calls/nx:Call', ns)
        if call_nodes:
            for idx, call in enumerate(call_nodes):
                stop_point_ref = call.find('nx:ScheduledStopPointRef', ns)
                if stop_point_ref is None:
                    continue

                stop_id = stop_point_ref.get('ref', '')
                arrival = call.find('nx:ArrivalTime', ns)
                departure = call.find('nx:DepartureTime', ns)
                time_str = (departure.text if departure is not None and departure.text
                            else arrival.text if arrival is not None and arrival.text
                            else None)

                if stop_id and time_str:
                    journey_stops.append({
                        'stop_id': stop_id,
                        'sequence': idx,
                        'time': time_str,
                    })

        # Variant B: passingTimes/TimetabledPassingTime
        if not journey_stops:
            passing_nodes = journey.findall('nx:passingTimes/nx:TimetabledPassingTime', ns)
            for idx, passing in enumerate(passing_nodes):
                spjp_ref_elem = passing.find('nx:StopPointInJourneyPatternRef', ns)
                stop_id = None
                if spjp_ref_elem is not None:
                    spjp_ref = spjp_ref_elem.get('ref', '')
                    stop_id = spjp_to_stop_id.get(spjp_ref)

                if not stop_id:
                    direct_stop_ref = passing.find('nx:ScheduledStopPointRef', ns)
                    if direct_stop_ref is not None:
                        stop_id = direct_stop_ref.get('ref', '')

                arrival = passing.find('nx:ArrivalTime', ns)
                departure = passing.find('nx:DepartureTime', ns)
                time_str = (departure.text if departure is not None and departure.text
                            else arrival.text if arrival is not None and arrival.text
                            else None)

                if stop_id and time_str:
                    journey_stops.append({
                        'stop_id': stop_id,
                        'sequence': idx,
                        'time': time_str,
                    })

        if journey_stops:
            result['journeys'].append({
                'journey_id': journey_id,
                'stops': journey_stops,
            })

    return result if result['line_number'] else None

print("✓ NeTEx XML parser defined (Call + TimetabledPassingTime support)")

✓ NeTEx XML parser defined (Call + TimetabledPassingTime support)


## Section 4: Load Train Data from ZIP

In [4]:
# Load all train data from DSB operators
train_data = {}  # line_number -> parsed data
date_range = {'min': None, 'max': None}

print(f"Loading {len(TRAIN_OPERATORS)} train operators from NeTEx ZIP...")
print("-" * 60)

with zipfile.ZipFile(zip_path) as z:
    for operator in sorted(TRAIN_OPERATORS):
        # Find all NeTEx files for this operator
        operator_files = [f for f in z.namelist() if f.startswith(operator + '/')]
        
        for file_path in operator_files:
            try:
                xml_content = z.read(file_path).decode('utf-8')
                parsed = parse_netex_xml(xml_content, operator)
                
                if parsed and parsed['line_number']:
                    line = parsed['line_number']
                    
                    # Store or merge with existing line data
                    if line not in train_data:
                        train_data[line] = parsed
                    else:
                        # Merge stops and journeys
                        train_data[line]['stops'].update(parsed['stops'])
                        train_data[line]['journeys'].extend(parsed['journeys'])
                    
                    # Track date range
                    if parsed['valid_from']:
                        if date_range['min'] is None or parsed['valid_from'] < date_range['min']:
                            date_range['min'] = parsed['valid_from']
                    if parsed['valid_to']:
                        if date_range['max'] is None or parsed['valid_to'] > date_range['max']:
                            date_range['max'] = parsed['valid_to']
                            
            except Exception as e:
                # Silently skip files that can't be parsed
                pass

print(f"✓ Loaded {len(train_data)} train lines")
print(f"✓ Date range: {date_range['min']} to {date_range['max']}")
print(f"\nTrain lines available:")
for line in sorted(train_data.keys()):
    data = train_data[line]
    print(f"  Line {line}: {len(data['stops'])} stops, {len(data['journeys'])} journeys")

Loading 5 train operators from NeTEx ZIP...
------------------------------------------------------------
✓ Loaded 24 train lines
✓ Date range: 2026-02-27 to 2026-05-20

Train lines available:
  Line 11746: 266 stops, 1949 journeys
  Line 11747: 204 stops, 825 journeys
  Line 11748: 308 stops, 2995 journeys
  Line 16448: 37 stops, 851 journeys
  Line 16449: 29 stops, 943 journeys
  Line 16451: 32 stops, 948 journeys
  Line 16452: 47 stops, 456 journeys
  Line 16453: 15 stops, 1096 journeys
  Line 16454: 26 stops, 125 journeys
  Line 21848: 96 stops, 63 journeys
  Line 22369: 8 stops, 235 journeys
  Line 23838: 50 stops, 175 journeys
  Line 23839: 105 stops, 197 journeys
  Line 23888: 85 stops, 223 journeys
  Line 23918: 38 stops, 45 journeys
  Line 23919: 14 stops, 123 journeys
  Line 23921: 43 stops, 43 journeys
  Line 24403: 2 stops, 16 journeys
  Line 24559: 32 stops, 91 journeys
  Line 24609: 23 stops, 67 journeys
  Line 26199: 52 stops, 68 journeys
  Line 26269: 6 stops, 206 journe

## Section 5: Interactive Date Selector

In [5]:
from ipywidgets import DatePicker, VBox, HBox, HTML, Button, Output
from IPython.display import display

# Parse date range
min_date = datetime.strptime(date_range['min'], '%Y-%m-%d').date() if date_range['min'] else datetime.now().date()
max_date = datetime.strptime(date_range['max'], '%Y-%m-%d').date() if date_range['max'] else datetime.now().date()
default_date = min_date

# Create date picker widget
date_picker = DatePicker(
    value=default_date,
    description='Select Date:',
    disabled=False,
    min_date=min_date,
    max_date=max_date
)

# Display info and date picker
info_html = HTML(f"""
<div style="padding: 10px; background-color: #f0f0f0; border-radius: 5px; margin-bottom: 10px;">
    <b>Train Schedule Data</b><br>
    Available dates: <code>{date_range['min']}</code> to <code>{date_range['max']}</code><br>
    Select a date below to visualize trains for that day
</div>
""")

display(info_html)
display(date_picker)

print(f"✓ Date picker ready (default: {default_date})")

HTML(value='\n<div style="padding: 10px; background-color: #f0f0f0; border-radius: 5px; margin-bottom: 10px;">…

DatePicker(value=datetime.date(2026, 2, 27), description='Select Date:', step=1)

✓ Date picker ready (default: 2026-02-27)


## Section 6: Build Train Movement Simulation (1 hour = 10 seconds)

This section defines the core simulation logic. Key functions:

- **`get_line_color(line_number)`**: Returns a stable color for each line (same color on every run)
- **`is_excluded_service(line_data)`**: Filters out S-trains and metro lines using metadata
- **`parse_time_to_seconds(raw_time)`**: Converts ISO 8601 times to seconds since midnight
- **`build_train_segments_for_date(selected_date)`**: Builds interpolatable movement segments for all trains on a date
- **`interpolate_train_position(segment, simulation_sec)`**: Calculates exact train position between two stops
- **`collect_active_trains(segments, simulation_sec)`**: Finds all trains between stops at a given time
- **`create_base_map_with_optional_stations(...)`**: Creates the MapLibre map with optional station layer
- **`update_train_markers(m, active_trains, previous_marker_ids)`**: Updates train positions on map in-place

The simulation uses **linear interpolation**: A train moving from Stop A to Stop B over 900 seconds is positioned proportionally to elapsed time.

For architecture details, see [docs/train_simulation.md](../docs/train_simulation.md).

In [6]:
import time
import hashlib


def get_line_color(line_number: str) -> str:
    """Return a stable color for a train line."""
    colors = [
        "#FF6B6B", "#4ECDC4", "#45B7D1", "#FFA07A", "#98D8C8",
        "#F7DC6F", "#BB8FCE", "#85C1E2", "#F8B739", "#52C0A1",
    ]
    digest = hashlib.md5(line_number.encode("utf-8")).hexdigest()
    return colors[int(digest[:8], 16) % len(colors)]


def is_excluded_service(line_data: Dict) -> bool:
    """Return True for S-train and metro services we do not want to simulate."""
    operator_name = (line_data.get("operator_name") or "").lower()
    line_public_code = (line_data.get("line_public_code") or "").strip()
    line_name = (line_data.get("line_name") or "").strip()
    transport_mode = (line_data.get("line_transport_mode") or "").lower()

    # Metro exclusion
    if transport_mode == "metro":
        return True
    if line_public_code.upper().startswith("M") and len(line_public_code) <= 3:
        return True
    if line_name.upper().startswith("M") and len(line_name) <= 3:
        return True

    # S-train exclusion
    if "s-tog" in operator_name or "s tog" in operator_name:
        return True

    # Common S-train line labels/codes
    s_train_codes = {
        "A", "B", "Bx", "C", "E", "F", "H",
        "BLÅ", "BLAA", "RØD", "ROED", "GRØN", "GROEN", "GUL", "LILLA", "ORANGE", "PINK",
    }
    if line_public_code in s_train_codes:
        return True
    if line_name in s_train_codes:
        return True

    # Some S-train replacement feeds use togbus naming
    if "togbus" in line_name.lower():
        return True

    return False


def parse_time_to_seconds(raw_time: Optional[str]) -> Optional[int]:
    """Parse a NeTEx time string into seconds since midnight."""
    if not raw_time:
        return None

    text = raw_time.strip()

    if "T" in text:
        text = text.split("T", 1)[1]

    if "+" in text:
        text = text.split("+", 1)[0]
    if "-" in text and text.count(":") >= 2 and text.rfind("-") > text.rfind(":"):
        text = text.rsplit("-", 1)[0]
    if text.endswith("Z"):
        text = text[:-1]

    parts = text.split(":")
    if len(parts) < 2:
        return None

    try:
        hours = int(parts[0])
        minutes = int(parts[1])
        seconds = int(parts[2]) if len(parts) > 2 else 0
    except ValueError:
        return None

    return hours * 3600 + minutes * 60 + seconds


def build_train_segments_for_date(selected_date):
    """Build interpolatable train movement segments for the selected date."""
    segments = []
    used_lines = set()

    for line_number, line_data in train_data.items():
        if is_excluded_service(line_data):
            continue

        if line_data.get("valid_from") and line_data.get("valid_to"):
            valid_from = datetime.strptime(line_data["valid_from"], "%Y-%m-%d").date()
            valid_to = datetime.strptime(line_data["valid_to"], "%Y-%m-%d").date()
            if not (valid_from <= selected_date <= valid_to):
                continue

        stop_lookup = line_data.get("stops", {})

        for journey in line_data.get("journeys", []):
            journey_stops = journey.get("stops", [])
            if len(journey_stops) < 2:
                continue

            points = []
            for call in journey_stops:
                stop_id = call.get("stop_id")
                stop_info = stop_lookup.get(stop_id)
                seconds_since_midnight = parse_time_to_seconds(call.get("time"))
                if not stop_info or seconds_since_midnight is None:
                    continue
                points.append({
                    "time_sec": seconds_since_midnight,
                    "lat": stop_info["lat"],
                    "lng": stop_info["lng"],
                    "stop_name": stop_info["name"],
                })

            if len(points) < 2:
                continue

            points.sort(key=lambda item: item["time_sec"])

            destination = points[-1]["stop_name"]
            line_color = get_line_color(line_number)

            for idx in range(len(points) - 1):
                start = points[idx]
                end = points[idx + 1]
                if end["time_sec"] <= start["time_sec"]:
                    continue

                segments.append({
                    "train_id": f"{line_number}::{journey.get('journey_id', idx)}",
                    "line": line_number,
                    "color": line_color,
                    "destination": destination,
                    "start_sec": start["time_sec"],
                    "end_sec": end["time_sec"],
                    "start_lat": start["lat"],
                    "start_lng": start["lng"],
                    "end_lat": end["lat"],
                    "end_lng": end["lng"],
                })

            used_lines.add(line_number)

    return segments, used_lines


def interpolate_train_position(segment: Dict, simulation_sec: int) -> Tuple[float, float]:
    """Interpolate train position between two timed stops."""
    span = segment["end_sec"] - segment["start_sec"]
    if span <= 0:
        return segment["start_lng"], segment["start_lat"]

    ratio = (simulation_sec - segment["start_sec"]) / span
    ratio = max(0.0, min(1.0, ratio))

    lng = segment["start_lng"] + (segment["end_lng"] - segment["start_lng"]) * ratio
    lat = segment["start_lat"] + (segment["end_lat"] - segment["start_lat"]) * ratio
    return lng, lat


def collect_active_trains(segments: List[Dict], simulation_sec: int) -> List[Dict]:
    """Collect train positions active at the given simulation time."""
    active = {}

    for segment in segments:
        if not (segment["start_sec"] <= simulation_sec < segment["end_sec"]):
            continue

        train_id = segment["train_id"]
        lng, lat = interpolate_train_position(segment, simulation_sec)

        active[train_id] = {
            "train_id": train_id,
            "line": segment["line"],
            "destination": segment["destination"],
            "color": segment["color"],
            "lng": lng,
            "lat": lat,
        }

    return list(active.values())


def create_base_map_with_optional_stations(show_stations_layer=True):
    """Create map and optionally draw train stations as an extra layer."""
    m = LiveMapLibreMap(center=(12.5683, 55.6761), zoom=7, height="650px", width="100%")
    m.add_basemap("OpenStreetMap.Mapnik")

    station_count = 0
    if show_stations_layer:
        seen_station_names = set()
        for line_data in train_data.values():
            if is_excluded_service(line_data):
                continue
            for stop in line_data.get("stops", {}).values():
                station_name = stop.get("name", "")
                if not station_name or station_name in seen_station_names:
                    continue
                seen_station_names.add(station_name)
                m.add_marker(
                    lng=stop["lng"],
                    lat=stop["lat"],
                    name=f"station-{station_count}",
                    color="#9AA0A6",
                    popup=f"Station: {station_name}",
                )
                station_count += 1

    return m, station_count


def update_train_markers(m, active_trains: List[Dict], previous_marker_ids: List[str]) -> List[str]:
    """Update train markers in-place using LiveMapLibreMap.move_marker."""
    current_ids = []

    for train in active_trains:
        marker_id = train["train_id"]
        m.move_marker(
            marker_id,
            (train["lng"], train["lat"]),
            color=train["color"],
            popup=(
                f"<b>Train line {train['line']}</b><br>"
                f"Destination: {train['destination']}<br>"
                f"Position time: simulated"
            ),
        )
        current_ids.append(marker_id)

    for marker_id in previous_marker_ids:
        if marker_id not in current_ids:
            try:
                m.remove_marker(marker_id)
            except Exception:
                pass

    return current_ids


print("✓ Simulation helpers ready (LiveMapLibreMap.move_marker)")

✓ Simulation helpers ready (LiveMapLibreMap.move_marker)


## Section 7: Run Day Simulation with Interactive Controls

This cell sets up **interactive buttons** to control the simulation playback:

- **Start**: Load data for the selected date and begin the simulation
- **Pause**: Freeze the simulation in place
- **Resume**: Continue from where you paused
- **Stop**: End the simulation and clear the map

### Settings (Adjust These to Customize)

```python
SECONDS_PER_SIM_HOUR = 10      # Speed: 1 sim hour = 10 real seconds (240× speedup)
UPDATE_INTERVAL_REAL_SEC = 1   # Map refresh rate: update every 1 real second
SHOW_STATIONS_LAYER = False    # Show station dots on map (disabled to reduce clutter)
```

### How It Works

1. When you click **Start**, the code:
   - Builds movement segments for the selected date
   - Creates a MapLibre map centered on Copenhagen
   - Spawns a background thread to run the simulation loop

2. The simulation loop:
   - Runs every 1 real second
   - Calculates current simulated time (incremented by `sim_seconds_per_real_second`)
   - Collects active trains (those between stops at this time)
   - Updates all train markers in-place using `LiveMapLibreMap.move_marker()`
   - Updates the status display with current time and train count

3. **Pause/Resume** uses a flag to pause the background thread without stopping it

4. **Stop** sets the running flag to False, allowing the thread to exit cleanly

The entire 24-hour day takes approximately 240 real seconds (4 minutes) to simulate at these settings.

In [7]:
import threading
from IPython.display import display

# --- Simulation settings ---
SECONDS_PER_SIM_HOUR = 10   # 1 simulated hour = 10 real seconds
UPDATE_INTERVAL_REAL_SEC = 1
SHOW_STATIONS_LAYER = False  # disabled to avoid map clutter

# --- UI controls ---
start_button = Button(description="Start", button_style="success")
pause_button = Button(description="Pause", button_style="warning")
resume_button = Button(description="Resume", button_style="info")
stop_button = Button(description="Stop", button_style="danger")
status_html = HTML("<b>Status:</b> Ready")
log_output = Output()

controls = HBox([start_button, pause_button, resume_button, stop_button])
display(controls)
display(status_html)
display(log_output)

# --- Shared simulation state ---
simulation_running = False
simulation_paused = False
simulation_thread = None
simulation_map = None
simulation_segments = []
simulation_markers = []


def _set_status(text: str):
    status_html.value = f"<b>Status:</b> {text}"


def _run_simulation_loop():
    global simulation_running, simulation_paused, simulation_markers

    total_real_seconds = 24 * SECONDS_PER_SIM_HOUR
    sim_seconds_per_real_second = 3600 / SECONDS_PER_SIM_HOUR

    with log_output:
        print("Simulation started")
        print(f"Speed: 1 simulated hour = {SECONDS_PER_SIM_HOUR} real seconds")
        print(f"Duration: {total_real_seconds} seconds for full day")

    for real_second in range(0, total_real_seconds, UPDATE_INTERVAL_REAL_SEC):
        if not simulation_running:
            break

        while simulation_paused and simulation_running:
            time.sleep(0.2)

        if not simulation_running:
            break

        simulation_sec = int(real_second * sim_seconds_per_real_second)
        simulation_sec = min(simulation_sec, 24 * 3600 - 1)

        active_trains = collect_active_trains(simulation_segments, simulation_sec)
        simulation_markers = update_train_markers(
            simulation_map,
            active_trains,
            simulation_markers,
        )

        if real_second % 10 == 0:
            hh = simulation_sec // 3600
            mm = (simulation_sec % 3600) // 60
            ss = simulation_sec % 60
            _set_status(
                f"Running | simulated {hh:02d}:{mm:02d}:{ss:02d} | active trains: {len(active_trains)}"
            )

        time.sleep(UPDATE_INTERVAL_REAL_SEC)

    if simulation_running:
        _set_status("Completed full day simulation")
        with log_output:
            print("Simulation completed")
    else:
        _set_status("Stopped")
        with log_output:
            print("Simulation stopped")

    simulation_running = False
    simulation_paused = False


def on_start_clicked(_):
    global simulation_running, simulation_paused, simulation_thread
    global simulation_map, simulation_segments, simulation_markers

    if simulation_running:
        _set_status("Already running")
        return

    selected_date = date_picker.value
    if selected_date is None:
        _set_status("Select a date first")
        return

    segments, used_lines = build_train_segments_for_date(selected_date)
    if not segments:
        _set_status("No train movement data for selected date")
        return

    simulation_map, station_count = create_base_map_with_optional_stations(
        show_stations_layer=SHOW_STATIONS_LAYER,
    )
    display(simulation_map)

    simulation_running = True
    simulation_paused = False
    simulation_segments = segments
    simulation_markers = []

    with log_output:
        print(f"Date: {selected_date}")
        print(f"Lines included: {len(used_lines)}")
        print(f"Movement segments: {len(segments)}")
        print(f"Stations layer markers: {station_count}")
        print("Using LiveMapLibreMap.move_marker for train updates")

    _set_status("Running")
    simulation_thread = threading.Thread(target=_run_simulation_loop, daemon=True)
    simulation_thread.start()


def on_pause_clicked(_):
    global simulation_paused

    if not simulation_running:
        _set_status("Not running")
        return
    simulation_paused = True
    _set_status("Paused")


def on_resume_clicked(_):
    global simulation_paused

    if not simulation_running:
        _set_status("Not running")
        return
    simulation_paused = False
    _set_status("Running")


def on_stop_clicked(_):
    global simulation_running, simulation_paused

    if not simulation_running:
        _set_status("Not running")
        return
    simulation_running = False
    simulation_paused = False
    _set_status("Stopping...")


start_button.on_click(on_start_clicked)
pause_button.on_click(on_pause_clicked)
resume_button.on_click(on_resume_clicked)
stop_button.on_click(on_stop_clicked)

print("✓ Controls ready: Start / Pause / Resume / Stop")

HTML(value='<b>Status:</b> Ready')

Output()

✓ Controls ready: Start / Pause / Resume / Stop
